Partie 1 – Explorer les données 

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [11]:
#Chargement des données CSV
df = pd.read_csv("../data/smart_building_raw.csv")
print("Dimensions :", df.shape)

Dimensions : (507, 14)


In [12]:
#Affichage des premières lignes du dataset
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


In [13]:
#Affichage desS dernières lignes du dataset
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


In [14]:
# Point 5 : combien de variables (colonnes) possède le dataset ?
# df.shape[1] correspond au nombre de colonnes
print(f"Nombre de variables : {df.shape[1]}")

Nombre de variables : 14


In [15]:
# Point 6 : identifier les variables numeriques
# On utilise df.dtypes pour voir le type de chaque colonne,
# puis on liste manuellement celles qui sont vraiment numeriques (mesures continues)
print(df.dtypes)


id_mesure               int64
date                   object
batiment               object
type_batiment          object
zone                   object
temperature           float64
humidite              float64
co2                   float64
occupation            float64
consommation_kwh      float64
mode_climatisation     object
etat_systeme           object
jour_semaine           object
alerte                 object
dtype: object


In [16]:
# On definit explicitement la liste des variables numeriques du dataset
variables_numeriques = ["temperature", "humidite", "co2", "occupation", "consommation_kwh"]
print("Variables numeriques :", variables_numeriques)

Variables numeriques : ['temperature', 'humidite', 'co2', 'occupation', 'consommation_kwh']


In [17]:
# Point 7 : identifier les variables categorielles
# Ce sont les colonnes qui representent des categories (texte, classes), pas des mesures continues
variables_categorielles = ["batiment", "type_batiment", "zone", "mode_climatisation", "etat_systeme", "jour_semaine", "alerte"]
print("Variables categorielles :", variables_categorielles)

Variables categorielles : ['batiment', 'type_batiment', 'zone', 'mode_climatisation', 'etat_systeme', 'jour_semaine', 'alerte']


In [18]:
# Point 8 : identifier la ou les variables de type date
variable_date = "date"
print("Variable de type date :", variable_date)
print(df[variable_date].head())

Variable de type date : date
0    2025-02-13 06:00:00
1    2025-03-10 12:00:00
2    2025-05-04 00:00:00
3    2025-01-19 00:00:00
4    2025-04-24 06:00:00
Name: date, dtype: object


In [19]:
# Point 9 : identifier la ou les variables qui servent d'identifiant
variable_id = "id_mesure"
print("Variable identifiant :", variable_id)
print("Nombre de valeurs uniques :", df[variable_id].nunique())
print("Nombre total de lignes    :", df.shape[0])

Variable identifiant : id_mesure
Nombre de valeurs uniques : 500
Nombre total de lignes    : 507


In [20]:
# Point 10 : statistiques descriptives des variables numeriques
# describe() calcule automatiquement : count, mean, std, min, 25%, 50% (mediane), 75%, max
df[variables_numeriques].describe()

,temperature,humidite,co2,occupation,consommation_kwh
count,495.000000,496.000000,500.000000,501.000000,502.000000
mean,24.154141,57.864113,844.150000,44.850299,169.069323
std,7.418465,16.026336,582.181386,24.949139,53.164294
min,-30.000000,-12.000000,89.000000,-20.000000,-100.000000
25%,21.600000,49.275000,623.750000,27.000000,136.875000
50%,24.000000,57.550000,787.500000,46.000000,169.800000
75%,26.000000,65.750000,952.000000,61.000000,202.975000
max,96.000000,160.000000,6000.000000,116.000000,336.200000


## Point 11 — Variables potentiellement problématiques

D'après les statistiques descriptives (point 10) et un premier coup d'œil sur les données :

- **temperature** : le minimum et le maximum semblent très extrêmes pour un bâtiment (ex. valeurs négatives fortes ou supérieures à 70°C) → probablement des erreurs de capteur.
- **humidite** : l'humidité relative doit être comprise entre 0 et 100 % ; or on observe des valeurs négatives et des valeurs bien au-dessus de 100 → incohérent physiquement.
- **co2** : certaines valeurs atteignent plusieurs milliers de ppm, largement au-dessus des niveaux réalistes en intérieur → suspect.
- **consommation_kwh** : présence de valeurs négatives, alors qu'une consommation ne peut pas être négative.
- **occupation** : présence de valeurs négatives, alors qu'un nombre de personnes ne peut pas être négatif.
- **type_batiment**, **mode_climatisation**, **jour_semaine** : catégories mal orthographiées ou avec une casse incohérente (ex. "École" / "ÉCOLE" / "ecole").
- **date** : certaines valeurs ne sont pas des dates valides (ex. "date_invalide", "31/99/2025").

Ces problèmes seront traités un par un dans les points suivants (incohérences, valeurs manquantes, doublons, valeurs aberrantes).